# 🧠 Implementing Diffusion-Style Denoising Models with Keras


## 📋 Overview

I build a denoising model on MNIST that follows the same core idea behind diffusion models: corrupt an image with noise, then train a network to recover the clean image from the corrupted version. I preprocess the data, wire up a convolutional encoder-decoder, batch it through a `tf.data` pipeline, train it with early stopping, evaluate the denoising quality visually, fine-tune by unfreezing layers, and then run three practice variations — changing the noise level, deepening the architecture, and comparing denoising performance across multiple noise levels.

Coming from RF, I keep mapping this onto channel noise and receiver design: noise corrupting a transmitted signal, and a receiver trying to recover the original transmission. The overarching theme of every section here is the same rate-distortion / signal-recovery story I've been building on throughout this course.

**What I cover:**
- 📥 Preprocessing MNIST for a convolutional denoiser
- 🏗️ Building an encoder → bottleneck → decoder with Conv2D / Conv2DTranspose
- 🔄 Batching with a `tf.data` pipeline (cache, batch, prefetch)
- 🎯 Training with early stopping
- 📊 Evaluating denoising quality visually
- 🧊 Fine-tuning by freezing/unfreezing layers
- 🧪 Practice: noise factor sensitivity, deeper architectures, multi-level noise comparison


## 🧩 Theory

### What a real diffusion model does

A diffusion model learns to reverse a **gradual, multi-step noising process**. The forward process adds a little Gaussian noise at each of $T$ steps until the image becomes pure noise:

$$
q(x_t \mid x_{t-1}) = \mathcal{N}\!\left(x_t;\ \sqrt{1-\beta_t}\, x_{t-1},\ \beta_t I\right), \qquad
x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)
$$

where $\beta_t$ is a noise schedule and $\bar{\alpha}_t = \prod_{s=1}^t (1-\beta_s)$. The network is trained to predict the noise $\epsilon$ that was added at each step, and generation happens by starting from pure noise and running the learned reverse process step by step:

$$
\mathcal{L} = \mathbb{E}_{x_0,\, \epsilon,\, t}\left[\, \lVert \epsilon - \epsilon_\theta(x_t, t) \rVert^2 \,\right]
$$

### What I actually build in this notebook

This notebook implements a **simplified, single-step** version: one fixed noise level, no timestep conditioning $t$, and the network maps noisy input directly to the clean image in a single forward pass rather than iterating over $T$ reverse steps.

$$
\hat{x}_0 = f_\theta(\tilde{x}), \qquad \tilde{x} = x_0 + \sigma\,\epsilon, \quad \epsilon \sim \mathcal{N}(0, I), \qquad \mathcal{L} = \text{MSE}(x_0, \hat{x}_0) \ \text{or}\ \text{BCE}(x_0, \hat{x}_0)
$$

| | Full diffusion model | This notebook |
|---|---|---|
| Noise process | $T$ incremental steps, schedule $\beta_t$ | One fixed noise level $\sigma$ |
| Model input | Noisy image **+ timestep** $t$ | Noisy image only |
| Training target | Predict the noise $\epsilon$ | Predict the clean image $x_0$ directly |
| Generation | Iterative reverse sampling from pure noise | Single forward pass |

It's really a **convolutional denoising autoencoder** — the same reconstruction idea from earlier in this course, just with images treated as 2D tensors (Conv2D/Conv2DTranspose) instead of flattened vectors. I still call out where it connects to genuine diffusion models below, since the connection is real even if the implementation is simplified.

### 📡 Telecom analogy

| Concept | Signal processing equivalent |
|---|---|
| Forward noising ($\beta_t$ schedule) | Progressive SNR degradation across repeated hops/channels |
| Reverse denoising | A receiver reconstructing the transmitted signal from a noisy received one |
| Single-step denoiser (this notebook) | One-shot equalization — a single correction pass, not iterative refinement |
| Iterative reverse diffusion (real models) | Multi-pass iterative decoding (e.g. turbo/LDPC codes refining an estimate over iterations) |
| `tf.data` batching/prefetch | Pipelining and double-buffering in a DSP chain — keep the processing pipeline fed without stalls |


## Part 1 — 📥 Data Preprocessing

I load MNIST, normalize pixels to $[0, 1]$, and expand each image to a `(28, 28, 1)` tensor — the shape a `Conv2D`-based model expects (unlike the flattened `Dense` autoencoder from earlier, this one keeps the spatial structure of the image). Then I corrupt both the train and test sets with Gaussian noise and clip back to a valid pixel range, since that noisy version is what the model will learn to clean up.


In [ ]:
%%capture
!pip install tensorflow-cpu==2.16.2

import os
# Suppress oneDNN optimizations and lower TensorFlow logging level
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'


In [ ]:
!pip install numpy


In [ ]:
!pip install matplotlib


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import Input, Conv2D, Flatten, Dense, Reshape, Conv2DTranspose
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

# Load the dataset
(x_train, _), (x_test, _) = mnist.load_data()

# Normalize the pixel values
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

# Expand dimensions to match the (28, 28, 1) input shape a Conv2D model expects
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

# Add Gaussian noise to simulate the forward noising process
noise_factor = 0.5
x_train_noisy = x_train + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_train.shape)
x_test_noisy = x_test + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_test.shape)

# Clip back to a valid pixel range [0, 1]
x_train_noisy = np.clip(x_train_noisy, 0., 1.)
x_test_noisy = np.clip(x_test_noisy, 0., 1.)


**What just happened:**

| Step | Purpose |
|---|---|
| Normalize to $[0,1]$ | Stable, fast convergence during training |
| Expand to `(28,28,1)` | Match the 3D tensor shape Conv2D layers expect |
| Add Gaussian noise | Create the corrupted input the model will learn to reverse |
| Clip to $[0,1]$ | Keep pixel values valid after adding noise |


## Part 2 — 🏗️ Building the Denoising Model

I wire up an encoder (two `Conv2D` layers, then flatten into a dense bottleneck) and a decoder (dense expansion, reshape, two `Conv2DTranspose` layers) that together map a noisy `(28,28,1)` image back to a clean one. Compiled with Adam and MSE, since I'm regressing continuous pixel values rather than doing binary classification per pixel.


In [ ]:
# Define the diffusion-style denoising model architecture with reduced complexity
input_layer = Input(shape=(28, 28, 1))
x = Conv2D(16, (3, 3), activation='relu', padding='same')(input_layer)  # Reduced filters
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)  # Reduced filters
x = Flatten()(x)
x = Dense(64, activation='relu')(x)  # Reduced size
x = Dense(28*28*32, activation='relu')(x)  # Reduced size
x = Reshape((28, 28, 32))(x)
x = Conv2DTranspose(32, (3, 3), activation='relu', padding='same')(x)  # Reduced filters
x = Conv2DTranspose(16, (3, 3), activation='relu', padding='same')(x)  # Reduced filters
output_layer = Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)
diffusion_model = Model(input_layer, output_layer)

# Compile the model with MSE, since this is effectively a pixel-regression task
diffusion_model.compile(optimizer='adam', loss='mean_squared_error')

# Summary of the model
diffusion_model.summary()


**Architecture summary:**

| Layer | Role | Filters/Units | Activation |
|---|---|---|---|
| Input | Noisy image | (28,28,1) | — |
| Conv2D ×2 | Encoder — extract spatial features | 16 → 32 | ReLU |
| Flatten + Dense | Bottleneck | 64 | ReLU |
| Dense + Reshape | Expand back to spatial tensor | 28×28×32 | ReLU |
| Conv2DTranspose ×2 | Decoder — rebuild spatial detail | 32 → 16 | ReLU |
| Conv2D (output) | Final reconstruction | 1 | Sigmoid |


## Part 3 — 🔄 Batching with a `tf.data` Pipeline

Instead of feeding raw NumPy arrays straight into `fit()`, I wrap the noisy/clean pairs in a `tf.data.Dataset`, cache them in memory after the first pass, batch them, and prefetch the next batch while the current one is still training. This is the same pipelining idea as double-buffering in a DSP system — keep the next chunk of data ready so the GPU/CPU never stalls waiting on I/O.


In [ ]:
# Cache and prefetch the data using TensorFlow data pipelines for faster loading
train_dataset = tf.data.Dataset.from_tensor_slices((x_train_noisy, x_train))
train_dataset = train_dataset.cache().batch(64).prefetch(tf.data.AUTOTUNE)  # Reduced batch size

val_dataset = tf.data.Dataset.from_tensor_slices((x_test_noisy, x_test))
val_dataset = val_dataset.cache().batch(64).prefetch(tf.data.AUTOTUNE)  # Reduced batch size


## Part 4 — 🎯 Training with Early Stopping

I train on the noisy → clean pairs and use `EarlyStopping` to halt training once validation loss stops improving for 2 consecutive epochs, restoring the best weights seen. This is a built-in stopping criterion — similar to an iterative decoder stopping once its error estimate converges rather than running a fixed number of iterations regardless of whether they're still helping.


In [ ]:
# Implement early stopping based on validation loss
early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

# Train the model with early stopping and a smaller batch size
diffusion_model.fit(
    train_dataset,
    epochs=3,
    shuffle=True,
    validation_data=val_dataset,
    callbacks=[early_stopping]
)


## Part 5 — 📊 Evaluating Denoising Performance

I run the noisy test images through the trained model and compare original, noisy, and denoised versions side by side — the three-way comparison makes it easy to judge how much of the corruption the model actually removed.


In [ ]:
import matplotlib.pyplot as plt

# Predict the denoised images
denoised_images = diffusion_model.predict(x_test_noisy)

# Visualize the results
n = 10  # number of digits to display
plt.figure(figsize=(20, 6))
for i in range(n):
    # Original
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # Noisy
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(x_test_noisy[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # Denoised
    ax = plt.subplot(3, n, i + 1 + 2*n)
    plt.imshow(denoised_images[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
plt.show()


**Reading the result:** top = original, middle = noisy input, bottom = denoised output. The closer the bottom row sits to the top row, the more effectively the model reversed the noise corruption in a single pass.


## Part 6 — 🧊 Fine-Tuning by Freezing/Unfreezing Layers

Same pattern as the standard-autoencoder notebook: freeze every layer, confirm the frozen state, unfreeze just the last four (the decoder side), recompile, and retrain. This time I recompile with binary cross-entropy instead of MSE — a reminder that recompiling after changing `trainable` flags is also a chance to reconsider the loss function, not just refresh the optimizer state.


In [ ]:
# Freeze all the layers
for layer in diffusion_model.layers:
    layer.trainable = False


In [ ]:
# Check trainable status of each layer
for i, layer in enumerate(diffusion_model.layers):
    print(f"Layer {i}: {layer.name} — Trainable: {layer.trainable}")


In [ ]:
# Unfreeze the top layers of the model
for layer in diffusion_model.layers[-4:]:
    layer.trainable = True

# Recompile the model
diffusion_model.compile(optimizer='adam', loss='binary_crossentropy')

# Train the model again
diffusion_model.fit(x_train_noisy, x_train,
                    epochs=10,
                    batch_size=64,
                    shuffle=True,
                    validation_data=(x_test_noisy, x_test))


## 🎯 Practice: Noise Factor Sensitivity

**Objective:** see how a lower noise level changes the denoising task. I drop `noise_factor` from 0.5 to 0.3, regenerate the noisy datasets, and retrain — a smaller-magnitude noise term should be an easier signal-recovery problem, similar to improving the SNR of a noisy channel.


In [ ]:
# Change the noise factor to 0.3
noise_factor = 0.3

# Add noise to the data with the new noise factor
x_train_noisy = x_train + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_train.shape)
x_test_noisy = x_test + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_test.shape)

# Clip the values to be within the range [0, 1]
x_train_noisy = np.clip(x_train_noisy, 0., 1.)
x_test_noisy = np.clip(x_test_noisy, 0., 1.)

# Retrain the model
diffusion_model.fit(x_train_noisy, x_train,
                    epochs=50,
                    batch_size=128,
                    shuffle=True,
                    validation_data=(x_test_noisy, x_test))


## 🏗️ Practice: Adding More Layers

**Objective:** see how a deeper encoder/decoder affects denoising quality. I add a third `Conv2D` layer (128 filters) to the encoder and a matching `Conv2DTranspose` layer to the decoder, then rebuild, compile, and train from scratch.


In [ ]:
# Define the modified diffusion model architecture with additional layers
input_layer = Input(shape=(28, 28, 1))

x = Conv2D(32, (3, 3), activation='relu', padding='same')(input_layer)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)  # Additional layer
x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dense(28*28*64, activation='relu')(x)
x = Reshape((28, 28, 64))(x)
x = Conv2DTranspose(128, (3, 3), activation='relu', padding='same')(x)  # Additional layer
x = Conv2DTranspose(64, (3, 3), activation='relu', padding='same')(x)
x = Conv2DTranspose(32, (3, 3), activation='relu', padding='same')(x)
output_layer = Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

diffusion_model = Model(input_layer, output_layer)

# Compile the model
diffusion_model.compile(optimizer='adam', loss='binary_crossentropy')

# Summary of the model
diffusion_model.summary()

# Train the model
diffusion_model.fit(x_train_noisy, x_train,
                    epochs=50,
                    batch_size=128,
                    shuffle=True,
                    validation_data=(x_test_noisy, x_test))


## 🔍 Practice: Multi-Level Noise Comparison

**Objective:** compare denoising quality across several noise levels (0.1, 0.5, 0.7) side by side — the visual equivalent of plotting performance across a range of SNR values for a receiver.


In [ ]:
import matplotlib.pyplot as plt

# Function to add noise and predict denoised images
def add_noise_and_predict(noise_factor):
    x_test_noisy = x_test + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_test.shape)
    x_test_noisy = np.clip(x_test_noisy, 0., 1.)
    denoised_images = diffusion_model.predict(x_test_noisy)
    return x_test_noisy, denoised_images

# Noise levels to test
noise_levels = [0.1, 0.5, 0.7]

# Visualize the results
n = 5  # number of digits to display
plt.figure(figsize=(20, 12))
for idx, noise_factor in enumerate(noise_levels):
    x_test_noisy, denoised_images = add_noise_and_predict(noise_factor)

    for i in range(n):
        # Original
        ax = plt.subplot(3 * len(noise_levels), n, i + 1 + idx * 3 * n)
        plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)

        if i == 0:
            ax.set_title(f'Original (Noise: {noise_factor})')

        # Noisy
        ax = plt.subplot(3 * len(noise_levels), n, i + 1 + n + idx * 3 * n)
        plt.imshow(x_test_noisy[i].reshape(28, 28), cmap='gray')
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)

        # Denoised
        ax = plt.subplot(3 * len(noise_levels), n, i + 1 + 2 * n + idx * 3 * n)
        plt.imshow(denoised_images[i].reshape(28, 28), cmap='gray')
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
plt.show()


**What I expect:** at noise 0.1 the denoised output should look nearly identical to the original; at 0.7 the model has much less signal to work with, so reconstructions should visibly degrade — the same "SNR gets worse, error rate climbs" shape as a receiver's BER-vs-SNR curve.


## 📊 Summary

| Concept | What I did | Why it matters |
|---|---|---|
| 📥 Preprocessing | Normalized to $[0,1]$, expanded to `(28,28,1)`, added Gaussian noise | Sets up the noisy-input → clean-target denoising task |
| 🏗️ Conv2D architecture | Encoder/decoder with Conv2D + Conv2DTranspose | Preserves spatial structure, unlike the flattened Dense autoencoder |
| 🔄 `tf.data` pipeline | `cache().batch().prefetch(AUTOTUNE)` | Keeps training fed without I/O stalls |
| 🎯 Early stopping | `EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)` | Stops training once validation loss plateaus, restores the best weights |
| 📊 Evaluation | Original vs. noisy vs. denoised visual comparison | Direct read on how much corruption the model removed |
| 🧊 Fine-tuning | Froze all layers, unfroze last 4, recompiled with a different loss, retrained | Same transfer-learning pattern as before, plus a loss-function swap |
| 🎯 Noise sensitivity | Reduced noise factor to 0.3 and retrained | Lower noise = easier recovery, same SNR intuition |
| 🏗️ Deeper architecture | Added a third Conv2D/Conv2DTranspose layer pair | More capacity — worth testing whether it actually improves reconstruction |
| 🔍 Multi-level comparison | Compared denoising at noise 0.1 / 0.5 / 0.7 | Visual BER-vs-SNR-style comparison across corruption levels |

**Diffusion connection:** this notebook is a single-step, fixed-noise simplification of a real diffusion model — no timestep conditioning, no iterative reverse sampling. The theory section above spells out exactly what's missing (the $\beta_t$ schedule, the noise-prediction objective, the $T$-step reverse chain) so I don't walk away thinking this *is* a diffusion model rather than a stepping stone toward one.


## 🧪 Sandbox

Space to keep experimenting beyond the practice exercises:

- Add a timestep input and train the model to predict noise at several fixed noise levels — a first step toward real timestep conditioning
- Implement an actual noise schedule ($\beta_t$ increasing over $T$ steps) and see what it takes to chain single-step denoisers into a multi-step reverse process
- Swap MSE for the noise-prediction loss $\lVert \epsilon - \epsilon_\theta(x_t, t) \rVert^2$ and compare training behavior
- Apply this denoising pattern to a synthetic noisy RF waveform instead of MNIST digits and see if the same architecture transfers
- Compare this Conv2D denoiser against the Dense-based autoencoder from the earlier notebook on the same noise levels


In [ ]:
# 🧪 Sandbox — experiment here
